# ARE Benchmark — Modelos Locais via Ollama

Replica o benchmark [ARE (An R Eval)](https://github.com/diegoamrg4123/are-dataset-csv) usando modelos locais OSS via Ollama.  
O juiz é também um modelo local — sem dependência de APIs externas.

**Dependências:** `pip install ollama pandas`  
**Pré-requisito:** Ollama rodando (`ollama serve`) com os modelos instalados.

In [ ]:
import pandas as pd
import ollama
import json
from pathlib import Path
from datetime import datetime
from IPython.display import display

## Configuração
Edite `MODELS` e `JUDGE_MODEL` conforme os modelos instalados (`ollama list`).

In [ ]:
MODELS = [
    "lfm2.5-thinking:1.2b",
    "qwen3.5:2b",
    "qwen2.5-coder:3b",
    "gemma4:e2b",
]

JUDGE_MODEL = "nemotron-3-nano:4b"

EPOCHS = 3

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

## Dataset

In [ ]:
url = "https://raw.githubusercontent.com/diegoamrg4123/are-dataset-csv/main/are_dataset.csv"
df = pd.read_csv(url, sep=";", encoding="utf-8")
print(f"{len(df)} problemas carregados")
display(df.head(2))

## Solver
Envia o enunciado para o modelo avaliado e retorna a resposta.

In [ ]:
SOLVER_SYSTEM = "You are an expert R programmer. Answer concisely with working R code."

def solve(model: str, problem: str) -> str:
    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": SOLVER_SYSTEM},
            {"role": "user",   "content": problem}
        ]
    )
    return response["message"]["content"]

## Juiz / Scorer (partial credit)
Replica `model_graded_qa(partial_credit=TRUE)` do script R original.  
O juiz classifica cada resposta como **C** (correto), **P** (parcial) ou **I** (incorreto).

In [ ]:
JUDGE_PROMPT = """\
You are an expert R programmer evaluating an answer to an R coding task.

## Question
{question}

## Expected answer / grading rubric
{target}

## Model response
{response}

Grade the response with a single letter:
- C  (correct)  — fully achieves the task
- P  (partial)  — on the right track but incomplete or has minor errors
- I  (incorrect) — wrong or completely off

Reply with ONLY the letter C, P, or I."""

SCORE_MAP = {"C": 1.0, "P": 0.5, "I": 0.0}

def judge(question: str, target: str, response: str) -> float:
    result = ollama.chat(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": JUDGE_PROMPT.format(
            question=question, target=target, response=response
        )}]
    )
    letter = result["message"]["content"].strip().upper()[0]
    return SCORE_MAP.get(letter, 0.0)

## Runner
Itera sobre todos os problemas × epochs para um modelo. Pula se o resultado já existir.

In [ ]:
def run_benchmark(model: str, overwrite: bool = False):
    safe_name = model.replace(":", "_").replace("/", "_")
    result_path = RESULTS_DIR / f"{safe_name}.json"

    if not overwrite and result_path.exists():
        print(f"Skipping {model} — já existe em {result_path}")
        return

    records = []
    total = len(df) * EPOCHS
    done = 0

    for epoch in range(1, EPOCHS + 1):
        for _, row in df.iterrows():
            done += 1
            print(f"[{model}] epoch {epoch}/{EPOCHS} | {row['id']} ({done}/{total})", end=" ... ")
            response = solve(model, row["input"])
            score    = judge(row["input"], row["target"], response)
            records.append({
                "model":    model,
                "epoch":    epoch,
                "id":       row["id"],
                "domain":   row["domain"],
                "task":     row["task"],
                "score":    score,
                "response": response,
                "ts":       datetime.now().isoformat()
            })
            print(f"score={score}")

    result_path.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")
    mean_score = sum(r["score"] for r in records) / len(records)
    print(f"\n{model} — mean score: {mean_score:.3f} | salvo em {result_path}\n")

## Execução
Roda o benchmark para todos os modelos em `MODELS`. Modelos já avaliados são pulados automaticamente.

In [ ]:
for model in MODELS:
    run_benchmark(model)

## Análise de Resultados

In [ ]:
all_records = []
for p in RESULTS_DIR.glob("*.json"):
    all_records.extend(json.loads(p.read_text(encoding="utf-8")))

if not all_records:
    print("Nenhum resultado encontrado em results/. Execute o benchmark primeiro.")
else:
    results_df = pd.DataFrame(all_records)

    print("=== Score médio por modelo ===")
    summary = (
        results_df
        .groupby("model")["score"]
        .agg(mean="mean", std="std", n="count")
        .sort_values("mean", ascending=False)
        .round(3)
    )
    display(summary)

In [ ]:
if all_records:
    print("=== Score médio por modelo × domínio ===")
    domain_summary = (
        results_df
        .groupby(["model", "domain"])["score"]
        .mean()
        .unstack()
        .round(3)
    )
    display(domain_summary)

    print("\n=== Score médio por modelo × tipo de tarefa ===")
    task_summary = (
        results_df
        .groupby(["model", "task"])["score"]
        .mean()
        .unstack()
        .round(3)
    )
    display(task_summary)